In [0]:
caminho_csv = "/Volumes/dados-brutos/default/bucket-dados-brutos-churn/data.csv"

# Lê o CSV. O inferSchema tenta identificar automaticamente números, textos, etc.
df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load(caminho_csv)

# Cria a "Tabela Temporária"
df.createOrReplaceTempView("tabela_ecommerce")

print("Tabela carregada com sucesso!")

Tabela carregada com sucesso!


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW clientes_rfm AS
WITH base AS (
    SELECT 
        CustomerID,
        -- A correção acontece aqui: ensinamos que o texto é Mês/Dia/Ano Hora:Minuto
        MAX(to_timestamp(InvoiceDate, 'M/d/yyyy H:mm')) as UltimaCompra,
        COUNT(DISTINCT InvoiceNo) as Frequencia,
        SUM(Quantity * UnitPrice) as ValorMonetario
    FROM tabela_ecommerce
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID
)
SELECT 
    *,
    datediff((SELECT MAX(UltimaCompra) FROM base), UltimaCompra) as Recencia
FROM base;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dataset_final AS
SELECT 
    *,
    CASE WHEN Recencia > 90 THEN 'Churn (Perdido)' ELSE 'Ativo' END as Status_Cliente
FROM clientes_rfm;

In [0]:
# Consulta a tabela final e exibe na tela
df_final = spark.sql("SELECT * FROM dataset_final")
display(df_final)

CustomerID,UltimaCompra,Frequencia,ValorMonetario,Recencia,Status_Cliente
13408,2011-12-08T09:05:00.000Z,81,27487.409999999985,1,Ativo
14001,2011-10-25T10:31:00.000Z,4,2030.33,45,Ativo
12877,2011-12-06T10:30:00.000Z,12,1535.7700000000004,3,Ativo
13748,2011-09-05T09:45:00.000Z,5,948.25,95,Churn (Perdido)
16891,2011-12-08T11:15:00.000Z,5,809.6999999999998,1,Ativo
15012,2011-10-17T12:22:00.000Z,2,423.0399999999999,53,Ativo
13008,2011-01-20T14:24:00.000Z,2,178.47000000000003,323,Churn (Perdido)
14057,2011-11-16T10:32:00.000Z,18,6147.4000000000015,23,Ativo
12652,2011-01-21T17:01:00.000Z,2,914.5299999999997,322,Churn (Perdido)
15093,2011-11-21T13:09:00.000Z,9,4410.14,18,Ativo
